In [7]:
!pip install -q transformers accelerate peft bitsandbytes datasets trl

In [8]:
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model

In [9]:
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [10]:
model_name = "Qwen/Qwen1.5-1.8B"

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
print(tokenizer.pad_token, tokenizer.eos_token)

<|endoftext|> <|endoftext|>


In [12]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("BitsAndBytes 4-bit config created")

BitsAndBytes 4-bit config created


In [13]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False

print("Model loaded in 4-bit (QLoRA base model)")

model.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model loaded in 4-bit (QLoRA base model)


In [14]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)


In [15]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 1,839,974,400 || trainable%: 0.1710


In [18]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/train.jsonl",
        "validation": "/content/val.jsonl"
    }
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1283
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 143
    })
})


In [19]:
def format_prompt(example):
    prompt = (
        "### Instruction:\n"
        f"{example['instruction']}\n\n"
        "### Input:\n"
        f"{example['input']}\n\n"
        "### Response:\n"
        f"{example['output']}"
    )
    return prompt


def tokenize_function(example):
    prompt = format_prompt(example)
    return tokenizer(
        prompt,
        truncation=True,
        max_length=256,
        padding="max_length"
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names
)

print(tokenized_dataset)

Map:   0%|          | 0/1283 [00:00<?, ? examples/s]

Map:   0%|          | 0/143 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1283
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 143
    })
})


In [20]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/lora_outputs",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

print("TrainingArguments created successfully")

TrainingArguments created successfully


In [21]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator
)

print("Trainer created successfully")

Trainer created successfully


In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.056812,1.037558
2,0.791546,0.947817
3,0.683611,0.930742


TrainOutput(global_step=963, training_loss=0.9205568476753195, metrics={'train_runtime': 1001.4935, 'train_samples_per_second': 3.843, 'train_steps_per_second': 0.962, 'total_flos': 9038419442270208.0, 'train_loss': 0.9205568476753195, 'epoch': 3.0})

In [23]:
final_adapter_path = "/content/adapters"

trainer.model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)

print("Final LoRA adapters saved")

Final LoRA adapters saved


In [24]:
!zip -r adapters_qwen_day2.zip /content/adapters

  adding: content/adapters/ (stored 0%)
  adding: content/adapters/adapter_model.safetensors (deflated 8%)
  adding: content/adapters/chat_template.jinja (deflated 46%)
  adding: content/adapters/adapter_config.json (deflated 57%)
  adding: content/adapters/tokenizer.json (deflated 81%)
  adding: content/adapters/tokenizer_config.json (deflated 49%)
  adding: content/adapters/README.md (deflated 65%)
